# AQL Agent Demo

A smolagents-based agent that explores the **Archive Query Log (AQL)** — 357M search queries, 306M SERPs, 2.6B results across 25 years of web archives.

The agent connects to the AQL API server via **MCP (Model Context Protocol)**, which publishes all available tools automatically — no manual tool definitions needed.

## 1. Setup

In [ ]:
!pip install -Uq smolagents python-dotenv requests smolagents[mcp]

## 2. Configuration

Credentials are loaded from **`.env`** in this directory. The only value you need to fill in is `WEBIS_API_KEY`.
The Elasticsearch credentials in `.env` are read by the AQL API server (not this notebook directly).

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path().resolve() / ".env")

if not os.getenv("WEBIS_API_KEY"):
    print("⚠️  WEBIS_API_KEY not set — fill it in in .env")
else:
    print("✓ Credentials loaded")
    print(f"  Webis URL: {os.getenv('WEBIS_OPENAI_BASE_URL')}")

AQL_API_BASE = os.getenv("AQL_API_BASE", "https://aql-api-preview.web.webis.de")
print(f"  AQL API:   {AQL_API_BASE}")

## 3. AQL API

The AQL API is hosted publicly

- **API docs**: https://aql-api-preview.web.webis.de/docs
- **MCP endpoint**: https://aql-api-preview.web.webis.de/mcp/

In [ ]:
import requests

resp = requests.get(f"{AQL_API_BASE}/health", timeout=5)
assert resp.status_code == 200, f"API not reachable at {AQL_API_BASE} — is the server running?"
print(f"✓ AQL API is up at {AQL_API_BASE}")

## 4. Model Setup

We use smolagents' `OpenAIServerModel`, which calls any OpenAI-compatible API directly via the `openai` SDK.
The Webis Chat API is OpenAI-compatible, so no extra adapter layer is needed.

To swap to a different provider, change `api_base` and `model_id`:

In [ ]:
from smolagents import OpenAIServerModel

model = OpenAIServerModel(
    model_id="qwen3-30b-a3b",
    api_base=os.getenv("WEBIS_OPENAI_BASE_URL"),
    api_key=os.getenv("WEBIS_API_KEY"),
)

# Swap to a different provider by changing api_base and model_id:
# model = OpenAIServerModel(model_id="gpt-4o-mini")                                  # OpenAI
# model = OpenAIServerModel(model_id="claude-sonnet-4-6", api_base="https://...")    # Anthropic-compatible
# model = OpenAIServerModel(model_id="llama3", api_base="http://localhost:11434/v1") # Ollama

print(f"Using model: {model.model_id}")

---
## MCP Agent

The AQL API server exposes a **Model Context Protocol (MCP)** endpoint at `/mcp`.
smolagents' `MCPClient` connects to it and **discovers all available tools automatically** —
no manual tool definitions needed.

This is the key value of MCP: the server owns and publishes its tool definitions;
any compatible client (Claude Desktop, Cursor, smolagents, ...) can consume them without extra code.

The AQL MCP exposes all read endpoints (`search_serps`, `count_serps`, `suggest_serps_queries`,
`serp_date_histogram`, `search_providers`, `search_archives`, ...) —
write and monitoring routes are excluded automatically.

In [ ]:
import smolagents.models as _sm_models
from smolagents import MCPClient, ToolCallingAgent


def _clean_schema(s):
    """
    Recursively strip schema fields that OpenAI function calling rejects.

    FastMCP generates schemas with anyOf, title, format, etc.
    We whitelist only what OpenAI-compatible APIs accept.

    'nullable' is injected for optional parameters (nullable anyOf or explicit default)
    so smolagents' validate_tool_arguments accepts missing arguments. It is stripped
    from the API payload by the get_tool_json_schema patch below.
    """
    if not isinstance(s, dict):
        return s
    # Resolve anyOf → pick the non-null branch; inject nullable for optional params
    if "anyOf" in s:
        non_null = [x for x in s["anyOf"] if x.get("type") != "null"]
        has_null = any(x.get("type") == "null" for x in s["anyOf"])
        s = {**s, **(non_null[0] if non_null else {})}
        s.pop("anyOf", None)
        if has_null and not s.get("nullable"):
            s["nullable"] = True
    allowed = {"type", "description", "enum", "items", "properties", "required", "default", "nullable"}
    result = {k: v for k, v in s.items() if k in allowed}
    # Params with an explicit default are also optional
    if "default" in result:
        result.setdefault("nullable", True)
    if "properties" in result:
        result["properties"] = {k: _clean_schema(v) for k, v in result["properties"].items()}
    if "items" in result:
        result["items"] = _clean_schema(result["items"])
    return result


# Patch: strip 'nullable' from the API schema before it reaches the LLM.
# smolagents uses it internally (in t.inputs) but OpenAI-compatible APIs reject it.
_orig_get_tool_json_schema = _sm_models.get_tool_json_schema

def _get_tool_json_schema_clean(tool):
    result = _orig_get_tool_json_schema(tool)
    for prop in result["function"]["parameters"]["properties"].values():
        prop.pop("nullable", None)
    return result

_sm_models.get_tool_json_schema = _get_tool_json_schema_clean


def make_mcp_agent(max_steps: int = 10) -> ToolCallingAgent:
    """
    Create a ToolCallingAgent with a fresh MCP session.
    Always create a new instance per agent.run() — MCP sessions don't survive restarts.
    """
    client = MCPClient({"url": f"{AQL_API_BASE}/mcp/"}, structured_output=False)
    tools = client.get_tools()
    for t in tools:
        t.inputs = {name: _clean_schema(schema) for name, schema in t.inputs.items()}
    return ToolCallingAgent(tools=tools, model=model, max_steps=max_steps)


# Smoke-test: list discovered tools
_agent = make_mcp_agent()
print(f"MCP tools available: {[t.name for t in _agent.tools.values() if t.name != 'final_answer']}")

In [ ]:
make_mcp_agent().run(
    "Which search providers in AQL cover academic or scientific search? List their names."
)

In [ ]:
make_mcp_agent().run(
    "Find 5 example queries about 'Saxophone' in AQL. "
    "Which providers captured them and what years are they from?"
)

In [ ]:
make_mcp_agent(max_steps=15).run(
    "How has interest in 'Alto Saxophone' changed over time in AQL? "
    "Show yearly query counts and describe the trend. "
    "Find reasons why that could be, so what the queries were about in those years and what was happening in the world that could explain it. Be specific and name example queries, such as the most numerous. "
    "Make sure to investigate if there are outliers or anomalies in the trend and explain those as well."
)

---
## Optional — Gradio Chatbot UI

Interactive chat interface using the MCP agent (all AQL tools available).

In [ ]:
from smolagents import GradioUI

GradioUI(make_mcp_agent(), reset_agent_memory=True).launch()

---
## Your Turn

Three tiers — pick the depth that suits you.
All tiers use the same setup: `python app.py` opens the Gradio UI, or keep using this notebook.

### Tier 1 — Try it (15 min)

Open the Gradio UI and ask the agent each of these questions. Note which tool it calls each time.

1. *"How many SERPs are in AQL?"*
2. *"Which search engines have the most SERPs in the dataset?"*
3. *"How has interest in 'climate change' changed year by year?"*
4. One question of your own — about any topic you're curious about.

**Reflection:** Did the agent ever give an answer that surprised you? Was it grounded in the data or did it feel like a hallucination?

### Tier 2 — Add a custom tool (30–45 min)

Write a new `@tool`, add it alongside the MCP tools, and verify the agent uses it.

**Example:** a Wikipedia summary tool that lets the agent explain *why* a query spiked in a given year.

In [ ]:
from smolagents import tool, ToolCallingAgent, MCPClient


@tool
def wikipedia_summary(topic: str) -> str:
    """
    Fetch the opening paragraph of the Wikipedia article for a topic.
    Use this to provide background context for AQL query trends.

    Args:
        topic: The subject to look up (e.g. 'Bitcoin', 'COVID-19 pandemic').
    """
    import requests
    url = "https://en.wikipedia.org/api/rest_v1/page/summary/" + topic.replace(" ", "_")
    r = requests.get(url, timeout=10)
    return r.json().get("extract", "Not found") if r.ok else "Not found"


def make_mcp_agent_extended(max_steps: int = 12) -> ToolCallingAgent:
    client = MCPClient({"url": f"{AQL_API_BASE}/mcp/"}, structured_output=False)
    tools = client.get_tools()
    for t in tools:
        t.inputs = {name: _clean_schema(schema) for name, schema in t.inputs.items()}
    return ToolCallingAgent(
        tools=tools + [wikipedia_summary],  # <- add your tool here
        model=model,
        max_steps=max_steps,
    )


# Test your extended agent
make_mcp_agent_extended().run(
    "In the year with the most bitcoin queries in AQL, "
    "what was happening with bitcoin according to Wikipedia?"
)

**Other tool ideas for Tier 2:**
- A tool that fetches top news headlines for a date (e.g. via NewsAPI or a public RSS feed)
- A tool that reads a local CSV and returns a summary or row lookup
- A tool that queries Google Trends for a keyword
- A tool that returns archive availability stats from the Wayback Machine CDX API

Once your tool works, ask the agent a question that *requires* it — verify in the trace that it was actually called.

### Tier 3 — Connect a second MCP server (60–90 min)

Build a minimal FastMCP server for your own dataset and connect it alongside AQL.
The agent will then have access to *both* data sources and can reason across them.

In [ ]:
# Step 1: install FastMCP
# !pip install fastmcp

# Step 2: create my_server.py with something like:
#
#   from fastmcp import FastMCP
#   mcp = FastMCP("My Data")
#
#   @mcp.tool()
#   def get_event(year: int) -> str:
#       """Return a notable event for the given year."""
#       events = {2017: "Bitcoin bull run", 2020: "COVID-19 pandemic begins"}
#       return events.get(year, "No event recorded")
#
#   if __name__ == "__main__":
#       mcp.run(transport="sse", port=9000)

# Step 3: run it in a terminal:  python my_server.py

# Step 4: connect both MCP servers
from smolagents import MCPClient, ToolCallingAgent

def make_dual_agent(max_steps: int = 15) -> ToolCallingAgent:
    aql_client   = MCPClient({"url": f"{AQL_API_BASE}/mcp/"}, structured_output=False)
    my_client    = MCPClient({"url": "http://localhost:9000/mcp/"}, structured_output=False)
    aql_tools    = aql_client.get_tools()
    my_tools     = my_client.get_tools()
    for t in aql_tools + my_tools:
        t.inputs = {name: _clean_schema(schema) for name, schema in t.inputs.items()}
    return ToolCallingAgent(tools=aql_tools + my_tools, model=model, max_steps=max_steps)


# make_dual_agent().run(
#     "In the year with the most bitcoin queries in AQL, what major event happened?"
# )